# Grocery Store Sales - A project into data cleaning, analysis, and visualisations
### A retail transaction dataset containing 500 transactions and 15 columns. It represents grocery store sales activity across different stores, locations, product categories, products, payment methods, customer ratings, discounts, and returns.
*Source: https://www.kaggle.com/datasets/arjunsinghgangwar/grocery-store-sales-dataset*

I believe that this dataset has potential to create great visualisations and analysis, with the sizeable amount of columns and data types it has. Potential questions I am aiming to explore are: **DO NOT FORGET TO PUT THIS IN**

## Initial Data Exploration

In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
# import matplotlib.cm
import scipy
import squarify

In [25]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [26]:
file_path = "grocery_store_data.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "arjunsinghgangwar/grocery-store-sales-dataset",
  file_path
)

print("First 5 records:", df.head())

First 5 records:   Transaction_ID        Date Store_ID Store_Location Product_Category  \
0       TXN00305  2024-02-22       S4      Bangalore            Dairy   
1       TXN00341  2024-07-22       S2          Delhi          Produce   
2       TXN00048  2025-03-30       S1        Chennai          Produce   
3       TXN00068  2024-12-03       S1        chennai           Bakery   
4       TXN00480  2024-07-09       S1         Mumbai          Produce   

             Product_Name  Quantity_Sold  Unit_Price  Discount_Percent  \
0   Paneer - Local Brand               5       94.76               5.0   
1         Tomato - Nestle             15       46.41               NaN   
2           Tomato - Amul              3       29.47              20.0   
3         Bun - Britannia             17       31.24              15.0   
4            Tomato - HUL             32       46.30               5.0   

   Total_Sales Payment_Mode  Customer_Rating    Employee_Name  \
0       450.11          UPI       

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    500 non-null    str    
 1   Date              500 non-null    str    
 2   Store_ID          500 non-null    str    
 3   Store_Location    500 non-null    str    
 4   Product_Category  500 non-null    str    
 5   Product_Name      500 non-null    str    
 6   Quantity_Sold     500 non-null    int64  
 7   Unit_Price        500 non-null    float64
 8   Discount_Percent  460 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      500 non-null    str    
 11  Customer_Rating   470 non-null    float64
 12  Employee_Name     500 non-null    str    
 13  Contact           500 non-null    str    
 14  Returned          500 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 58.7 KB


In [28]:
df[['Product_Category', 'Total_Sales']].head()

,Product_Category,Total_Sales
0,Dairy,450.11
1,Produce,696.15
2,Produce,70.73
3,Bakery,451.42
4,Produce,1407.52


## Data Cleaning

In [29]:
# Bringing back the information on the dataframe:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    500 non-null    str    
 1   Date              500 non-null    str    
 2   Store_ID          500 non-null    str    
 3   Store_Location    500 non-null    str    
 4   Product_Category  500 non-null    str    
 5   Product_Name      500 non-null    str    
 6   Quantity_Sold     500 non-null    int64  
 7   Unit_Price        500 non-null    float64
 8   Discount_Percent  460 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      500 non-null    str    
 11  Customer_Rating   470 non-null    float64
 12  Employee_Name     500 non-null    str    
 13  Contact           500 non-null    str    
 14  Returned          500 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 58.7 KB


As we can see from viewing the dataframe information, there are 500 entries, but not all of these entries have non-null values
for each and every column. Whereas the data types all seem adequate for their respective column, the non-null values can be problematic before we start our analysis.

The key outliers and how many values are missing are the columns of:

* 'Discount_Percent' is missing 40 values.
* 'Total_Sales' is missing 15 values.
* 'Customer_Rating' is missing 30 values.

I will now go through the methods that I will use to clean all of them, as well as reasoning.

### Cleaning the 'Total_Sales' column

As this column only consists of 15 missing values, and a total sales column would be hard to estimate for each product, I feel comfortable in dropping the missing rows here:

In [30]:
df = df.dropna(subset=['Total_Sales']) # using dropna to drop only the null rows from Total Sales

In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 485 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    485 non-null    str    
 1   Date              485 non-null    str    
 2   Store_ID          485 non-null    str    
 3   Store_Location    485 non-null    str    
 4   Product_Category  485 non-null    str    
 5   Product_Name      485 non-null    str    
 6   Quantity_Sold     485 non-null    int64  
 7   Unit_Price        485 non-null    float64
 8   Discount_Percent  449 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      485 non-null    str    
 11  Customer_Rating   456 non-null    float64
 12  Employee_Name     485 non-null    str    
 13  Contact           485 non-null    str    
 14  Returned          485 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 60.6 KB


As we can see above, now the total number of entries is 485 from the 15 removed, and 'Total_Sales' is in line with that.
'Discount_Percent' now only has 36 missing values, and 'Customer_Rating' now only has 29 missing values.

### Cleaning the 'Discount_Percent' column

My idea for cleaning this column, is that instead of removing these rows are there are a lot of rows, I will try to find similar discounts for these empty rows compared to others of the same of either the product name/unit price, or the broader product category. This should give a close or exact potential discount figure.

In [32]:
# To find the 36 rows where the values are missing
missing_discounts = df[df['Discount_Percent'].isnull()]
missing_discounts[['Transaction_ID', 'Product_Category', 'Product_Name', 'Unit_Price', 'Discount_Percent']].head(10)

,Transaction_ID,Product_Category,Product_Name,Unit_Price,Discount_Percent
1,TXN00341,Produce,Tomato - Nestle,46.41,NaN
52,TXN00006,Frozen Foods,Frozen Pizza - Nestle,166.34,NaN
85,TXN00224,Beverages,Green Tea - Patanjali,71.60,NaN
86,TXN00445,Beverages,Cola - HUL,47.75,NaN
99,TXN00111,Dairy,Yogurt - Local Brand,70.90,NaN
100,TXN00239,Beverages,Orange Juice - Amul,71.69,NaN
104,TXN00381,Beverages,Cola - ITC,51.72,NaN
108,TXN00208,Produce,Apple - Nestle,43.61,NaN
109,TXN00102,Dairy,Milk - HUL,48.00,NaN
116,TXN00437,Snacks,Biscuits - Britannia,29.69,NaN


Looking at this table above, I am going to decide not to use the product category to fill out the discount percent. The reason for this is that products of the same category have vastly varying unit prices, which casts doubt over if their discounts would be similar or not.

In [33]:
df['Discount_Percent'] = df['Discount_Percent'].fillna(
    df.groupby('Product_Name')['Discount_Percent'].transform('median')
)

Using the fillna method to fill empty values, the code breaks down each product name into groups of the same product name, then takes the median discount for each group using the transform method, which gives every row's median inside their group.

In [34]:
df.info()

<class 'pandas.DataFrame'>
Index: 485 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    485 non-null    str    
 1   Date              485 non-null    str    
 2   Store_ID          485 non-null    str    
 3   Store_Location    485 non-null    str    
 4   Product_Category  485 non-null    str    
 5   Product_Name      485 non-null    str    
 6   Quantity_Sold     485 non-null    int64  
 7   Unit_Price        485 non-null    float64
 8   Discount_Percent  472 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      485 non-null    str    
 11  Customer_Rating   456 non-null    float64
 12  Employee_Name     485 non-null    str    
 13  Contact           485 non-null    str    
 14  Returned          485 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 60.6 KB


We see now that the discount percent column has gone from 449 non-null values, to 472 non-null values. This essentially means that there are 13 products that are completely unique and have no other transactions that can be compared to them to match their discounts. This gives us three main options: either I can just use an overall median discount for the unique products, I can fill them with no discount as an assumption, or I can remove those rows.

For the interest in row preservation, since this dataset is now less than 500 rows, I am opting to use the median of each product category to fill out the 13 missing values of discount percent, using the same method as the previous code:

In [35]:
df['Discount_Percent'] = df['Discount_Percent'].fillna(
    df.groupby('Product_Category')['Discount_Percent'].transform('median')
)

In [36]:
df.info()

<class 'pandas.DataFrame'>
Index: 485 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    485 non-null    str    
 1   Date              485 non-null    str    
 2   Store_ID          485 non-null    str    
 3   Store_Location    485 non-null    str    
 4   Product_Category  485 non-null    str    
 5   Product_Name      485 non-null    str    
 6   Quantity_Sold     485 non-null    int64  
 7   Unit_Price        485 non-null    float64
 8   Discount_Percent  485 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      485 non-null    str    
 11  Customer_Rating   456 non-null    float64
 12  Employee_Name     485 non-null    str    
 13  Contact           485 non-null    str    
 14  Returned          485 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 60.6 KB


Now that this has been fixed, there are only 29 rows left to clean.

## Cleaning the 'Customer_Rating' column

Again, due to my dataset being now under 500 rows, I wish not to limit this further. It could be said not to fill in the customer ratings if I was, say, a store owner as this would be ethically wrong, but from an analysis perspective I shall use the medians of their products, with the justification that the average customer ratings should not drastically change by this method with only a few extra ratings filled in for each product.

First, let's see what rating system this column uses:

In [37]:
df[['Transaction_ID', 'Product_Name', 'Customer_Rating']].head(10)

,Transaction_ID,Product_Name,Customer_Rating
0,TXN00305,Paneer - Local Brand,4.0
1,TXN00341,Tomato - Nestle,3.4
2,TXN00048,Tomato - Amul,2.6
3,TXN00068,Bun - Britannia,3.8
4,TXN00480,Tomato - HUL,2.9
5,TXN00486,Yogurt - Local Brand,3.0
6,TXN00311,Dish Soap - Patanjali,2.8
7,TXN00032,Butter - Britannia,4.0
8,TXN00250,Bread - Amul,3.3
9,TXN00091,Chips - Patanjali,3.6


This tells me a lot, it informs that the customer rating system is up to 5, and also that it may include decimals (which was listed in the .info() as the data type float64), which works for us in using and adding averages without drastically shifting the data. We can now apply the same logic as previously, but this time using the product name, as products that are the same should in theory be similar in customer rating.

In [38]:
df['Customer_Rating'] = df['Customer_Rating'].fillna(
    df.groupby('Product_Name')['Customer_Rating'].transform('median')
)

In [39]:
df.info()

<class 'pandas.DataFrame'>
Index: 485 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    485 non-null    str    
 1   Date              485 non-null    str    
 2   Store_ID          485 non-null    str    
 3   Store_Location    485 non-null    str    
 4   Product_Category  485 non-null    str    
 5   Product_Name      485 non-null    str    
 6   Quantity_Sold     485 non-null    int64  
 7   Unit_Price        485 non-null    float64
 8   Discount_Percent  485 non-null    float64
 9   Total_Sales       485 non-null    float64
 10  Payment_Mode      485 non-null    str    
 11  Customer_Rating   479 non-null    float64
 12  Employee_Name     485 non-null    str    
 13  Contact           485 non-null    str    
 14  Returned          485 non-null    str    
dtypes: float64(4), int64(1), str(10)
memory usage: 60.6 KB


Now there are only six values that are non-null in our dataset. As it is only six, we make the median being used into the broader group of product category. This also helps to keep the cleaning logic consistent throughout this stage.

In [40]:
df['Customer_Rating'] = df['Customer_Rating'].fillna(
    df.groupby('Product_Category')['Customer_Rating'].transform('median')
)

The final check:

In [41]:
print(df.isnull().sum())

Transaction_ID      0
Date                0
Store_ID            0
Store_Location      0
Product_Category    0
Product_Name        0
Quantity_Sold       0
Unit_Price          0
Discount_Percent    0
Total_Sales         0
Payment_Mode        0
Customer_Rating     0
Employee_Name       0
Contact             0
Returned            0
dtype: int64


Before ending, I just want to check whether the 'Transaction_ID' column, which as a primary key should be all unique values, has any duplicates. Other rows are allowed to be non-unique, but this column must be. This will be outputted as a true/false boolean.

In [42]:
print("Are there any duplicate Transaction IDs?", df['Transaction_ID'].duplicated().any())

Are there any duplicate Transaction IDs? False


There are no duplicates of issue. The data has been cleaned and is ready to analyse.

## Data Analysis

The following are five analytical questions I have created, as well as using Pandas to answer them each.

### Question 1 - What product categories generate the highest total revenue, and how does the total quantity sold compare?

It is important for the store to work out what areas of the store should be focused on as there is a higher demand/revenue for it, and thus what is generating the most amount of money for the store. To begin with, I need the dataframe grouped by product category, with the total sales and quantity sold for their corresponding category added together to get final values for each category:

In [46]:
category_performance = df.groupby('Product_Category')[['Total_Sales', 'Quantity_Sold']].sum().reset_index()
# the reset index method helps for presentation to put it in table format

# This next step just helps to sort by the total sales in descending order to see the top performers first
category_performance = category_performance.sort_values(by='Total_Sales', ascending=False)
category_performance

,Product_Category,Total_Sales,Quantity_Sold
3,Frozen Foods,245033.61,1585
2,Dairy,143856.08,2313
4,Household,138068.54,1036
1,Beverages,94411.51,1734
6,Snacks,70489.82,1926
5,Produce,69796.58,2159
0,Bakery,66528.13,1505


We can see that the frozen foods category generates the highest revenue of the store, followed by the dairy section.